# Revision 2 — UCI-HAR coalition budget sensitivity

Reviewer 1 comment 5: does the 20-epoch surrogate game change teacher rankings and the weights used for a final 60-epoch student?

This experiment freezes the three original teacher banks (11, 23, 37), split, cached teacher probability targets, architecture, optimizer, learning rate, batch size, original student seed rules, validation checkpoint criterion and patience. Each of the 32 coalitions is trained once up to 60 epochs. Its first 20 epochs supply the matched shorter-budget utility. Both budgets share the same training trajectory and numerical settings. Early stopping still applies; 60 is a limit.

**Research total: 96 coalition fits plus 6 matched final students.** The three extra final fits relative to the original 99-fit estimate provide a current-session 20-game baseline. The old published 20-epoch game is retained as a separately labeled historical comparison. Original results and models are preserved.

Coalition scoring uses validation data only. Both predeclared final students are evaluated on the already-used fixed test split as a post-review robustness analysis; test scores never choose weights or checkpoints. This does not produce new independent confirmatory evidence.

Run cells in order with a GPU. Outputs save incrementally to `MyDrive/makd-har/results/revision-2/uci-budget/`. The research cell checks smoke completion before starting. Restarting the cell resumes verified fits; a changed source, environment, seed or protocol stops reuse.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT=Path('/content/drive/MyDrive/makd-har')
EXPERIMENT=PROJECT/'revision_2/uci_budget'
assert (EXPERIMENT/'frozen-inputs.json').is_file(), 'Wait for Drive synchronization before continuing.'
print('Project:', PROJECT)


In [ ]:
# Use the installed Colab runtime; do not upgrade TensorFlow or Keras.
import subprocess, sys, json, os, datetime
import tensorflow as tf
print('TensorFlow',tf.__version__,'Keras',tf.keras.__version__)
print('GPU',tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Choose a GPU under Runtime > Change runtime type before continuing.')

def run_budget(*flags):
    logs=PROJECT/'results/revision-2/diagnostics'
    logs.mkdir(parents=True,exist_ok=True)
    label='verify' if '--verify-only' in flags else ('preflight' if '--preflight-only' in flags else ('smoke' if '--smoke' in flags else 'research'))
    log=logs/f"uci-budget-{label}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    command=[sys.executable,'-u',str(EXPERIMENT/'run_uci_budget.py'),'--drive-root',str(PROJECT),*flags]
    child_env=dict(os.environ,TF_CPP_MIN_LOG_LEVEL='2')
    print('Log:',log,flush=True)
    with log.open('w') as stream:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=child_env) as process:
            for line in process.stdout:
                stream.write(line);stream.flush();print(line,end='',flush=True)
            code=process.wait()
    if code:raise RuntimeError(f'{label} stopped with exit code {code}. The complete diagnostic is saved in {log}.')


In [ ]:
# File integrity and all three fixed-bank/split checks; no training.
run_budget('--verify-only')
run_budget('--preflight-only')


In [ ]:
# Separate smoke experiment: two teachers, four coalitions, two short final students.
# Smoke results are never included as research evidence.
run_budget('--smoke')
smoke=json.loads((PROJECT/'results/revision-2/smoke-uci-budget/completed.json').read_text())
assert smoke['status']=='complete' and smoke['mode']=='smoke' and smoke['fits']==6
print('SMOKE PASSED. Ready for the 102-fit research block.')


In [ ]:
# Research: all five teachers, three banks, 96 coalition fits + 6 matched final students.
smoke=json.loads((PROJECT/'results/revision-2/smoke-uci-budget/completed.json').read_text())
assert smoke['status']=='complete' and smoke['mode']=='smoke' and smoke['fits']==6
run_budget()


In [ ]:
completed=PROJECT/'results/revision-2/uci-budget/completed.json'
assert completed.is_file(), 'Research is incomplete. Rerun the research cell to resume verified results.'
record=json.loads(completed.read_text())
assert record['status']=='complete' and record['mode']=='research' and record['fits']==102
print('COMPLETE: 96 coalition fits and 6 final students. Ready for OneDrive analysis.')
print('Keep the saved results; no further rerun is needed.')


## Interpretation after completion

Compare the paired 20/60 games for raw Shapley magnitudes, ranks, lowest-value teacher, weight movement and final-student outcomes. Report agreement with the historical 20-epoch game separately. There are only three teacher banks; do not use this block to claim new independent subject-level significance. Keep null outcomes and do not alter seeds or stopping rules after observing them.
